# Final Project: Clean Execution Notebook

This notebook provides a simplified, isolated environment to execute the three submission runs step-by-step.

**Structure:**
1. **Setup**: Imports and Configuration.
2. **Helper Functions**: Fusion, Normalization, and Retrieval utilities.
3. **Run 1**: Baseline Fusion Execution.
4. **Run 2**: Structured Queries (SDM) Execution.
5. **Run 3**: Neural Reranking (MonoT5) Execution.

In [ ]:
# -----------------------------------------------------------------------------
# 1. SETUP & IMPORTS
# -----------------------------------------------------------------------------
import os
import json
import time
import math
from pathlib import Path
from collections import defaultdict
from typing import List, Dict, Tuple, Any

# Prevent Lucene memory issues
os.environ.setdefault("JAVA_TOOL_OPTIONS", "-Dorg.apache.lucene.store.MMapDirectory.enableMemorySegments=false")

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from pyserini.search.lucene import LuceneSearcher, LuceneImpactSearcher, LuceneHnswDenseSearcher
from pyserini.encode._splade import SpladeQueryEncoder
from pyserini.pyclass import autoclass

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Runtime Device: {DEVICE.upper()}")

# --- CONFIGURATION ---
PROJECT_ROOT = Path('.').resolve()
QUERIES_PATH = PROJECT_ROOT / 'Files-20260104' / 'queriesROBUST.txt'
HYDE_JSONL_PATH = PROJECT_ROOT / 'hyde_all_hypothetical_docs.jsonl'
QRELS_JUDGED_PATH = PROJECT_ROOT / 'Files-20260104' / 'qrels_50_Queries'

# Retrieval Constants
K = 1000
RM3_INDEX = 'robust04'
SPLADEPP_INDEX = 'beir-v1.0.0-robust04.splade-pp-ed'
SPLADEPP_MODEL = 'naver/splade-cocondenser-ensembledistil'
SPLADEV3_INDEX = 'beir-v1.0.0-robust04.splade-v3'
SPLADEV3_MODEL = 'naver/splade-v3-distilbert'
DENSE_INDEX = 'beir-v1.0.0-robust04.bge-base-en-v1.5.hnsw'
DENSE_ENCODER = 'BgeBaseEn15'

# MonoT5 Constants
MONOT5_MODEL = 'cramraj8/duqgen-monot5-3b-robust04-1k'
MONOT5_ALPHA = 0.3  # Interpolation weight (0.3 fusion + 0.7 monot5)
MONOT5_PASSAGE_CHARS = 1500
MONOT5_STRIDE = 1200
MONOT5_MAX_PASSAGES = 15

# Fusion Weights
W_RUN1 = [0.55, 0.10, 0.15, 0.20]  # RM3, SPLADE++, SPLADEv3, Dense
W_RUN2 = [0.60, 0.25, 0.15]        # SDM, SPLADE++, Dense

# Load Queries
def load_queries(path: Path) -> Dict[str, str]:
    qs = {}
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                qid, txt = line.strip().split('\t', 1)
                qs[qid] = txt
    return qs

all_queries = load_queries(QUERIES_PATH)
# Determine if we are running Judged (50) or Test (199). Default to TEST for final generation.
# Change to all_queries.keys()[:50] to run quick validation
target_qids = list(all_queries.keys())[50:] 
print(f"Loaded {len(all_queries)} queries. Targeted for this run: {len(target_qids)}")

# Load HyDE Docs
hyde_docs = {}
if HYDE_JSONL_PATH.exists():
    with HYDE_JSONL_PATH.open('r', encoding='utf-8') as f:
        for line in f:
            rec = json.loads(line)
            hyde_docs[str(rec['qid'])] = rec.get('text', '')
print(f"Loaded {len(hyde_docs)} HyDE documents")

def get_query_text(qid: str, mode: str) -> str:
    orig = all_queries[str(qid)]
    if mode == 'hyde':
        return orig + " " + hyde_docs.get(str(qid), "")
    return orig

In [ ]:
# -----------------------------------------------------------------------------
# 2. HELPER FUNCTIONS
# -----------------------------------------------------------------------------
def minmax_norm(scores: Dict[str, float]) -> Dict[str, float]:
    if not scores: return {}
    vals = list(scores.values())
    mn, mx = min(vals), max(vals)
    if mx - mn < 1e-9: return {d: 0.0 for d in scores}
    return {d: (s - mn) / (mx - mn) for d, s in scores.items()}

def fuse(run_dicts: List[Dict[str, float]], weights: List[float], k: int) -> List[Tuple[str, float]]:
    # Normalize inputs
    norms = [minmax_norm(d) for d in run_dicts]
    all_docs = set().union(*[d.keys() for d in norms])
    
    fused = {}
    for doc in all_docs:
        score = 0.0
        for w, n_dict in zip(weights, norms):
            score += w * n_dict.get(doc, 0.0)
        fused[doc] = score
    
    # Sort and cut
    ranked = sorted(fused.items(), key=lambda x: x[1], reverse=True)[:k]
    return ranked

def retrieve_one(searcher, query: str, k: int, generator=None) -> Dict[str, float]:
    if generator:
        hits = searcher.search(query, k=k, query_generator=generator)
    else:
        hits = searcher.search(query, k=k)
    return {h.docid: h.score for h in hits}

def write_run(path: Path, run_data: Dict[str, List[Tuple[str, float]]], tag: str):
    with path.open('w', encoding='utf-8') as f:
        for qid in sorted(run_data.keys(), key=int):
            for rank, (docid, score) in enumerate(run_data[qid], start=1):
                f.write(f"{qid} Q0 {docid} {rank} {score:.6f} {tag}\n")
    print(f"Written: {path}")

In [ ]:
# -----------------------------------------------------------------------------
# 3. RUN 1: BASELINE FUSION
# -----------------------------------------------------------------------------
# Initialize Searchers
print("Initializing Run 1 Searchers...")
searcher_rm3 = LuceneSearcher.from_prebuilt_index(RM3_INDEX)
searcher_rm3.set_bm25(0.9, 0.4)
searcher_rm3.set_rm3(20, 5, 0.5)

searcher_pp = LuceneImpactSearcher.from_prebuilt_index(SPLADEPP_INDEX, SpladeQueryEncoder(SPLADEPP_MODEL, device=DEVICE))
searcher_v3 = LuceneImpactSearcher.from_prebuilt_index(SPLADEV3_INDEX, SpladeQueryEncoder(SPLADEV3_MODEL, device=DEVICE))
searcher_dense = LuceneHnswDenseSearcher.from_prebuilt_index(DENSE_INDEX, encoder=DENSE_ENCODER, ef_search=1000)

run1_results = {}
print(f"Processing {len(target_qids)} queries for Run 1...")

for i, qid in enumerate(target_qids):
    q_txt_orig = get_query_text(qid, 'orig')
    q_txt_hyde = get_query_text(qid, 'hyde')
    
    # 1. Sparse
    s_rm3 = retrieve_one(searcher_rm3, q_txt_orig, K)
    # 2. SPLADE++
    s_pp = retrieve_one(searcher_pp, q_txt_orig, K)
    # 3. SPLADE-v3
    s_v3 = retrieve_one(searcher_v3, q_txt_orig, K)
    # 4. Dense (uses HyDE)
    s_dense = retrieve_one(searcher_dense, q_txt_hyde, K)
    
    # Fuse
    fused = fuse([s_rm3, s_pp, s_v3, s_dense], W_RUN1, K)
    run1_results[qid] = fused
    
    if (i+1) % 10 == 0: print(f"  Done {i+1}/{len(target_qids)}")

write_run(PROJECT_ROOT / 'run_1_clean.res', run1_results, 'run_1_fusion')

In [ ]:
# -----------------------------------------------------------------------------
# 4. RUN 2: STRUCTURED QUERIES (SDM)
# -----------------------------------------------------------------------------
print("Initializing Run 2 (SDM)...")
SdmQueryGenerator = autoclass('io.anserini.search.query.SdmQueryGenerator')
sdm_gen = SdmQueryGenerator(0.75, 0.10, 0.15)  # unigram, ordered, unordered

run2_results = {}
print(f"Processing {len(target_qids)} queries for Run 2...")

for i, qid in enumerate(target_qids):
    q_txt_orig = get_query_text(qid, 'orig')
    q_txt_hyde = get_query_text(qid, 'hyde')
    
    # 1. SDM + RM3 (Note: searcher_rm3 is already configured with RM3 params)
    # Passing the generator makes it an SDM query, RM3 expansion happens on top.
    s_sdm = retrieve_one(searcher_rm3, q_txt_orig, K, generator=sdm_gen)
    
    # 2. SPLADE++
    s_pp = retrieve_one(searcher_pp, q_txt_orig, K)
    
    # 3. Dense
    s_dense = retrieve_one(searcher_dense, q_txt_hyde, K)
    
    # Fuse [SDM, SPLADE, Dense]
    fused = fuse([s_sdm, s_pp, s_dense], W_RUN2, K)
    run2_results[qid] = fused
    
    if (i+1) % 10 == 0: print(f"  Done {i+1}/{len(target_qids)}")

write_run(PROJECT_ROOT / 'run_2_clean.res', run2_results, 'run_2_sdm')

In [ ]:
# -----------------------------------------------------------------------------
# 5. RUN 3: NEURAL RERANKING
# -----------------------------------------------------------------------------
print("Loading MonoT5...")
tokenizer = AutoTokenizer.from_pretrained(MONOT5_MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(MONOT5_MODEL, torch_dtype=torch.float16).to(DEVICE).eval()
true_id = tokenizer.encode('true', add_special_tokens=False)[0]
false_id = tokenizer.encode('false', add_special_tokens=False)[0]

def score_passages(query, text):
    # Split text
    passages = []
    for i in range(0, len(text), MONOT5_STRIDE):
        p = text[i : i + MONOT5_PASSAGE_CHARS]
        if len(p) > 10:
            passages.append(p)
        if len(passages) >= MONOT5_MAX_PASSAGES: break 
    if not passages: passages = [text[:MONOT5_PASSAGE_CHARS]]
    
    # Prepare Batch
    prompts = [f"Query: {query} Document: {p} Relevant:" for p in passages]
    inputs = tokenizer(prompts, return_tensors='pt', padding=True, truncation=True, max_length=512).to(DEVICE)
    
    with torch.no_grad():
        out = model.generate(input_ids=inputs.input_ids, attention_mask=inputs.attention_mask, max_new_tokens=1, return_dict_in_generate=True, output_scores=True)
    
    # Get scores (True - False logits)
    # This is a simplification; for exact logits we usually look at the stack. 
    # But for a clean notebook, using standard generation scores is often 'good enough' or we inspect scores[0]
    # For precision, let's use the explicit logit extraction pattern from the original code:
    
    decoder_input_ids = torch.full((len(prompts), 1), model.config.decoder_start_token_id, device=DEVICE)
    with torch.no_grad():
        logits = model(input_ids=inputs.input_ids, attention_mask=inputs.attention_mask, decoder_input_ids=decoder_input_ids).logits
    
    # shape: [batch, 1, vocab]
    token_logits = logits[:, 0, :]
    true_logits = token_logits[:, true_id]
    false_logits = token_logits[:, false_id]
    scores = (true_logits - false_logits).tolist()
    
    return max(scores) # MAXP Aggregation

run3_results = {}
print(f"Reranking {len(target_qids)} queries...")

# Note: Run 3 reranks the output of Run 1
for i, qid in enumerate(target_qids):
    query_txt = get_query_text(qid, 'orig')
    
    # Get candidate docs from Run 1 (Top 1000)
    candidates = run1_results.get(qid, []) # List of (docid, score)
    cand_docids = [d for d, s in candidates]
    
    # Fetch text (this is slow if not cached, but clean for notebook)
    monot5_scores = {}
    for docid in cand_docids:
        try:
            doc = searcher_rm3.doc(docid)
            raw = doc.raw() if doc else ""
            # Clean tags
            text = raw.replace("<HTML>", "").replace("</HTML>", "") # Simplified cleaning
            score = score_passages(query_txt, text)
            monot5_scores[docid] = score
        except Exception as e:
            monot5_scores[docid] = -100.0

    # Interpolate Scores
    # Normalize Run 1 scores
    r1_norm = minmax_norm({d: s for d, s in candidates})
    # Normalize MonoT5 scores
    t5_norm = minmax_norm(monot5_scores)
    
    final_scores = {}
    for docid in cand_docids:
        final_scores[docid] = MONOT5_ALPHA * r1_norm.get(docid, 0) + (1 - MONOT5_ALPHA) * t5_norm.get(docid, 0)
        
    # Sort
    ranked = sorted(final_scores.items(), key=lambda x: x[1], reverse=True)
    run3_results[qid] = ranked
    
    if (i+1) % 5 == 0: print(f"  Reranked {i+1}/{len(target_qids)}")

write_run(PROJECT_ROOT / 'run_3_clean.res', run3_results, 'run_3_neural')